# Tome S1, Chapter 8: Graphs through Their Eigenvalues (Python companion)

Exercises first, solutions at the end. We compute spectra and count walks, hunt for friendship graphs by brute force, sieve the parameter sets of strongly regular graphs, build the Hoffman–Singleton graph and check it, test Hoffman's bound, watch Graham–Pollak through the eyes of inertia, verify Huang's matrices and find the sharp example in $Q_4$, and see why three Petersen graphs do not fit into $K_{10}$. We use `numpy`, `matplotlib` and the standard library.

*Before each solution, predict what the output will look like.*

In [1]:
import itertools, math
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(8)

def spectrum(A, digits=6):
    ev = [float(e) + 0.0 for e in np.round(np.linalg.eigvalsh(A), digits)]
    return sorted(Counter(ev).items(), reverse=True)

def cube(n):
    N = 2 ** n
    return np.array([[1 if bin(x ^ y).count("1") == 1 else 0 for y in range(N)] for x in range(N)])

def petersen():
    V = list(itertools.combinations(range(5), 2))
    return np.array([[1 if not set(a) & set(b) else 0 for b in V] for a in V])

## Exercises

**P8.1 (★) Spectra and walks.** Compute the spectra of $K_6$, $C_8$, $Q_4$ and the Petersen graph and compare them with Example 8.1 and Exercise 8.1. Check that $\operatorname{tr}A^2=2|E|$ and $\operatorname{tr}A^3=6\cdot\#\text{triangles}$ for random graphs, and recover the answers of Exercise 8.2 from the spectrum alone.

**P8.2 (★★) Hunting friendship graphs.** Run through all graphs on $n\le6$ labelled vertices and keep those in which every two vertices have exactly one common neighbour. How many are there, and are they all windmills (Theorem 8.2)? Then check the arithmetic of Step 3 of the proof: for which $k\le 100$ is $k(k-1)+1$ vertices with $k+(r-s)\sqrt{k-1}=0$ solvable in integers?

**P8.3 (★★) The srg sieve.** List all parameter sets $(n,k,\lambda,\mu)$ with $n\le 40$, $0<k<n-1$, $\mu>0$, satisfying the counting identity 8.1 and the integrality of the multiplicities of Theorem 8.4. Then reproduce the Moore-graph computation of Theorem 8.5 for every $k\le 100$, and the multiplicities of the missing Moore graph (Exercise 8.4).

**P8.4 (★★) The Hoffman–Singleton graph.** Build it by Robertson's recipe: five pentagons $P_h$ (vertices $(h,i)$, $i\sim i\pm1$) and five pentagrams $Q_j$ (vertices $(j,k)$, $k\sim k\pm2$), with $(h,i)\in P_h$ joined to $(j,k)\in Q_j$ when $k\equiv hj+i\pmod 5$. Check that it is $\mathrm{srg}(50,7,0,1)$, that is, a Moore graph of degree $7$, and compute its spectrum. What does Hoffman's bound (Theorem 8.6) say about its independent sets? Find an independent set that large with a randomised greedy search.

**P8.5 (★) Kneser graphs and EKR.** Compute the spectrum of $K(n,2)$ for $n=5,\dots,9$, the Hoffman bound, and the true independence number by brute force for small $n$ (Example 8.7). Then look at Graham–Pollak (Theorem 8.8): decompose $K_n$ into bicliques in a few different ways and count the negative eigenvalues of each piece and of $J-I$ (Exercise 8.9).

**P8.6 (★★) Huang's matrices.** Build $A_n$ from the proof of Theorem 8.9, check $A_n^2=nI$ and that $|A_n|$ is the adjacency matrix of $Q_n$, and check that every $4$-cycle carries an odd number of minus signs (Figure 8.3). Then find, by brute force, the least possible maximum degree of an induced subgraph of $Q_n$ on $2^{n-1}+1$ vertices for $n\le4$, and the sharp examples in $Q_4$ (Exercise 8.11).

**P8.7 (★) Three Petersens.** Take two edge-disjoint copies of the Petersen graph in $K_{10}$, and compute the dimension of the intersection of their eigenspaces for the eigenvalue $1$ (Proposition 8.10). Then show numerically that the leftover graph $K_{10}-P-P'$ is never a Petersen graph, by computing its spectrum for many random pairs.

## Solutions

### P8.1
**How to find it.** `eigvalsh` for the spectra; for walks, the diagonal of $A^k$ counts closed walks.

In [2]:
K6 = np.ones((6, 6), int) - np.eye(6, dtype=int)
S = np.roll(np.eye(8, dtype=int), 1, axis=1); C8 = S + S.T
for name, A in [("K6", K6), ("C8", C8), ("Q4", cube(4)), ("Petersen", petersen())]:
    print(f"{name:9s}", spectrum(A))
print("C8 predicted:", sorted((round(2 * math.cos(2 * math.pi * j / 8), 6) + 0.0 for j in range(8)), reverse=True))

for trial in range(3):
    n = 9
    U = np.triu(rng.random((n, n)) < 0.4, 1).astype(int); A = U + U.T
    E = A.sum() // 2
    tri = sum(A[a, b] * A[b, c] * A[a, c] for a, b, c in itertools.combinations(range(n), 3))
    ev = np.linalg.eigvalsh(A)
    print(f"|E|={E}, tr A^2/2={np.sum(ev**2)/2:.4f};  triangles={tri}, tr A^3/6={np.sum(ev**3)/6:.4f}")

ev = np.array([3, 1, 1, 1, 1, 1, -2, -2, -2, -2])
print("spectrum 3,1^5,(-2)^4: n =", len(ev), " edges =", np.sum(ev**2) / 2, " triangles =", np.sum(ev**3) / 6,
      " average degree =", np.sum(ev**2) / len(ev), "= lambda_1, so regular")

K6        [(5.0, 1), (-1.0, 5)]
C8        [(2.0, 1), (1.414214, 2), (0.0, 2), (-1.414214, 2), (-2.0, 1)]
Q4        [(4.0, 1), (2.0, 4), (0.0, 6), (-2.0, 4), (-4.0, 1)]
Petersen  [(3.0, 1), (1.0, 5), (-2.0, 4)]
C8 predicted: [2.0, 1.414214, 1.414214, 0.0, 0.0, -1.414214, -1.414214, -2.0]
|E|=23, tr A^2/2=23.0000;  triangles=22, tr A^3/6=22.0000
|E|=12, tr A^2/2=12.0000;  triangles=2, tr A^3/6=2.0000
|E|=14, tr A^2/2=14.0000;  triangles=4, tr A^3/6=4.0000
spectrum 3,1^5,(-2)^4: n = 10  edges = 15.0  triangles = 0.0  average degree = 3.0 = lambda_1, so regular


The cycle's eigenvalues $2\cos(2\pi j/8)$ appear in pairs; the cube has $\binom4j$ copies of $4-2j$. The last line is Exercise 8.2: $\lambda_1$ equals the average degree only for regular graphs, so the mystery graph is $3$-regular with $15$ edges and no triangles. (It is the Petersen graph, but the spectrum alone does not prove that.)

### P8.2
**How to find it.** There are $2^{15}=32768$ graphs on $6$ labelled vertices: few enough to try them all with $A^2$ as a common-neighbour counter. A graph is a friendship graph exactly when $A^2$ has all off-diagonal entries equal to $1$.

In [3]:
def friendship_graphs(n):
    pairs = list(itertools.combinations(range(n), 2)); found = []
    for mask in range(1 << len(pairs)):
        A = np.zeros((n, n), int)
        for t, (a, b) in enumerate(pairs):
            if mask >> t & 1: A[a, b] = A[b, a] = 1
        A2 = A @ A
        if np.all(A2[~np.eye(n, dtype=bool)] == 1): found.append(A)
    return found

for n in range(2, 7):
    F = friendship_graphs(n)
    windmills = all(sorted(A.sum(0))[-1] == n - 1 and A.sum() // 2 == 3 * (n - 1) // 2 for A in F)
    print(f"n={n}: {len(F)} friendship graphs, all windmills: {windmills}")

print("windmills on 5 labelled vertices: 5 centres x 3 matchings =", 5 * 3)
sols = [k for k in range(1, 101) for r in range(k * (k - 1) + 1) if k * (k - 1) - r >= 0
        and abs(k + (r - (k * (k - 1) - r)) * math.sqrt(k - 1)) < 1e-9]
print("regular friendship spectra possible only for k =", sorted(set(sols)))

n=2: 0 friendship graphs, all windmills: True
n=3: 1 friendship graphs, all windmills: True
n=4: 0 friendship graphs, all windmills: True
n=5: 15 friendship graphs, all windmills: True


n=6: 0 friendship graphs, all windmills: True
windmills on 5 labelled vertices: 5 centres x 3 matchings = 15
regular friendship spectra possible only for k = [2]


Only $n=3$ and $n=5$ occur, with $1$ and $15$ labelled copies: a windmill on $5$ vertices is fixed by choosing the centre and a perfect matching of the other four. The arithmetic sieve leaves only $k=2$, the triangle, exactly as in Step 3. ($k=1$ fails because then $n=1$, $r=s=0$ and the trace would be $1$.)

### P8.3
**How to find it.** Loop over the four parameters; the multiplicities come from solving $f+g=n-1$, $k+f\theta+g\tau=0$.

In [4]:
def srg_feasible(n, k, l, m):
    if k * (k - l - 1) != (n - k - 1) * m: return None
    D = (l - m) ** 2 + 4 * (k - m)
    s = math.sqrt(D); th, ta = ((l - m) + s) / 2, ((l - m) - s) / 2
    f = (-k - (n - 1) * ta) / (th - ta); g = n - 1 - f
    if abs(f - round(f)) > 1e-9 or f < 0.5 or g < 0.5: return None
    return th, ta, round(f), round(g)

found = []
for n in range(5, 41):
    for k in range(2, n - 1):
        for l in range(0, k):
            for m in range(1, k + 1):
                r = srg_feasible(n, k, l, m)
                if r: found.append((n, k, l, m) + r)
print(len(found), "feasible parameter sets with n <= 40; the first ones:")
for t in found[:12]:
    print("  srg(%d,%d,%d,%d): eigenvalues %.3f, %.3f with multiplicities %d, %d" % t)

print("srg(21,10,4,5) passes the sieve:", srg_feasible(21, 10, 4, 5))
print("Moore graphs, degree k, n = k^2+1, srg(n,k,0,1):",
      [k for k in range(2, 101) if srg_feasible(k * k + 1, k, 0, 1)])
print("k = 57:", srg_feasible(57 ** 2 + 1, 57, 0, 1))

123 feasible parameter sets with n <= 40; the first ones:
  srg(5,2,0,1): eigenvalues 0.618, -1.618 with multiplicities 2, 2
  srg(6,3,0,3): eigenvalues 0.000, -3.000 with multiplicities 4, 1
  srg(6,4,2,4): eigenvalues 0.000, -2.000 with multiplicities 3, 2
  srg(8,4,0,4): eigenvalues 0.000, -4.000 with multiplicities 6, 1
  srg(8,6,4,6): eigenvalues 0.000, -2.000 with multiplicities 4, 3
  srg(9,4,1,2): eigenvalues 1.000, -2.000 with multiplicities 4, 4
  srg(9,6,3,6): eigenvalues 0.000, -3.000 with multiplicities 6, 2
  srg(10,3,0,1): eigenvalues 1.000, -2.000 with multiplicities 5, 4
  srg(10,5,0,5): eigenvalues 0.000, -5.000 with multiplicities 8, 1
  srg(10,6,3,4): eigenvalues 1.000, -2.000 with multiplicities 4, 5
  srg(10,8,6,8): eigenvalues 0.000, -2.000 with multiplicities 5, 4
  srg(12,6,0,6): eigenvalues 0.000, -6.000 with multiplicities 10, 1
srg(21,10,4,5) passes the sieve: (1.79128784747792, -2.79128784747792, 10, 10)
Moore graphs, degree k, n = k^2+1, srg(n,k,0,1): [2, 

Feasibility is necessary, not sufficient: the sieve keeps, for instance, the conference-type parameters $(n,\frac{n-1}2,\frac{n-5}4,\frac{n-1}4)$, and among those only some are realised (Paley graphs exist for prime powers $n\equiv1\pmod4$, and $n=21$ is feasible but impossible by finer arguments). The Moore sieve lets through exactly $2,3,7,57$, and the hypothetical graph of degree $57$ would have eigenvalues $7$ and $-8$ with multiplicities $1729$ and $1520$.

### P8.4
**How to find it.** Index vertex $(h,i)$ of $P_h$ as $5h+i$ and $(j,k)$ of $Q_j$ as $25+5j+k$, add the three kinds of edges, and test $A^2+A-6I=J$, which is the matrix form of $\mathrm{srg}(50,7,0,1)$.

In [5]:
HS = np.zeros((50, 50), int)
P = lambda h, i: 5 * h + i % 5
Q = lambda j, k: 25 + 5 * j + k % 5
for h in range(5):
    for i in range(5):
        HS[P(h, i), P(h, i + 1)] = HS[P(h, i + 1), P(h, i)] = 1
        HS[Q(h, i), Q(h, i + 2)] = HS[Q(h, i + 2), Q(h, i)] = 1
        for j in range(5):
            k = (h * j + i) % 5
            HS[P(h, i), Q(j, k)] = HS[Q(j, k), P(h, i)] = 1
J = np.ones((50, 50), int); I = np.eye(50, dtype=int)
print("7-regular:", set(HS.sum(0)), "  A^2 + A - 6I == J:", np.array_equal(HS @ HS + HS - 6 * I, J))
print("spectrum:", spectrum(HS))
lam_n = -3; print("Hoffman bound:", 50 * (-lam_n) / (7 - lam_n))

best = []
for trial in range(400):
    order = rng.permutation(50); chosen = []
    for v in order:
        if all(HS[v, w] == 0 for w in chosen): chosen.append(v)
    if len(chosen) > len(best): best = chosen
print("largest independent set found by random greedy:", len(best))

7-regular: {np.int64(7)}   A^2 + A - 6I == J: True
spectrum: [(7.0, 1), (2.0, 28), (-3.0, 21)]
Hoffman bound: 15.0
largest independent set found by random greedy: 15


The identity $A^2+A-6I=J$ says: $7$ closed walks of length $2$ at each vertex, no common neighbour for adjacent vertices, exactly one for non-adjacent ones. Hoffman's bound gives $\alpha\le15$, and the greedy search typically finds $15$ within a few hundred tries: the bound is sharp here, as it is for the Petersen graph ($10\cdot2/5=4$).

### P8.5
**How to find it.** Build $K(n,2)$ from pairs. The independence number of $K(n,2)$ is small enough to check by brute force over sets of pairs only for small $n$, so for $n\le6$ we check by brute force that no $n$ pairs are pairwise intersecting.

In [6]:
def kneser2(n):
    V = list(itertools.combinations(range(n), 2))
    return V, np.array([[1 if not set(a) & set(b) else 0 for b in V] for a in V])

def is_independent(S):
    return all(set(a) & set(b) for a, b in itertools.combinations(S, 2))

for n in range(5, 10):
    V, A = kneser2(n)
    ev = np.linalg.eigvalsh(A); k, lmin = ev[-1], ev[0]
    hoff = len(V) * (-lmin) / (k - lmin)
    if n <= 6:   # brute force: is there an intersecting family with n pairs?
        exists_n = any(is_independent(S) for S in itertools.combinations(V, n))
        extra = f", intersecting family of size n exists: {exists_n}"
    else:
        extra = ""
    print(f"K({n},2): spectrum {spectrum(A)}, Hoffman bound {hoff:.3f}, star size {n-1}{extra}")

K(5,2): spectrum [(3.0, 1), (1.0, 5), (-2.0, 4)], Hoffman bound 4.000, star size 4, intersecting family of size n exists: False
K(6,2): spectrum [(6.0, 1), (1.0, 9), (-3.0, 5)], Hoffman bound 5.000, star size 5, intersecting family of size n exists: False
K(7,2): spectrum [(10.0, 1), (1.0, 14), (-4.0, 6)], Hoffman bound 6.000, star size 6
K(8,2): spectrum [(15.0, 1), (1.0, 20), (-5.0, 7)], Hoffman bound 7.000, star size 7
K(9,2): spectrum [(21.0, 1), (1.0, 27), (-6.0, 8)], Hoffman bound 8.000, star size 8


The bound $n-1$ is attained by the star of pairs through a fixed point. For $n=5$ the triangle $\{12,13,23\}$ is another independent set, smaller than the star of size $4$.

For Graham–Pollak we count negative eigenvalues. Each biclique $K_{X,Y}$ has exactly one, and $J-I$ has $n-1$.

In [7]:
def biclique(n, X, Y):
    M = np.zeros((n, n), int)
    for x in X:
        for y in Y: M[x, y] = M[y, x] = 1
    return M

neg = lambda M: int(np.sum(np.linalg.eigvalsh(M) < -1e-9))

def random_decomposition(n):
    # split recursively: K_V = K_{V1,V2} + K_{V1} + K_{V2}
    parts, todo = [], [list(range(n))]
    while todo:
        V = todo.pop()
        if len(V) < 2: continue
        V = list(rng.permutation(V)); c = int(rng.integers(1, len(V)))
        parts.append((V[:c], V[c:])); todo += [V[:c], V[c:]]
    return parts

n = 9
target = np.ones((n, n), int) - np.eye(n, dtype=int)
print("negative eigenvalues of J - I:", neg(target))
for trial in range(4):
    parts = random_decomposition(n)
    total = sum(biclique(n, X, Y) for X, Y in parts)
    print(f"decomposition into {len(parts)} bicliques, exact: {np.array_equal(total, target)},"
          f" negative eigenvalues per piece: {[neg(biclique(n, X, Y)) for X, Y in parts]}")

negative eigenvalues of J - I: 8
decomposition into 8 bicliques, exact: True, negative eigenvalues per piece: [1, 1, 1, 1, 1, 1, 1, 1]
decomposition into 8 bicliques, exact: True, negative eigenvalues per piece: [1, 1, 1, 1, 1, 1, 1, 1]
decomposition into 8 bicliques, exact: True, negative eigenvalues per piece: [1, 1, 1, 1, 1, 1, 1, 1]
decomposition into 8 bicliques, exact: True, negative eigenvalues per piece: [1, 1, 1, 1, 1, 1, 1, 1]


Every recursive splitting uses exactly $n-1$ bicliques (each split adds one biclique and one more part, from $1$ part to $n$), and each piece contributes exactly one negative eigenvalue. Graham–Pollak says nothing cleverer is possible.

### P8.6
**How to find it.** Follow the recursion; a $4$-cycle of $Q_n$ flips two coordinates $i<j$, so loop over vertices and pairs of coordinates. For the extremal problem, $\binom{16}9=11440$ subsets is nothing.

In [8]:
def huang(n):
    A = np.array([[0, 1], [1, 0]])
    for m in range(2, n + 1):
        I = np.eye(len(A), dtype=int)
        A = np.block([[A, I], [I, -A]])
    return A

for n in range(1, 8):
    A = huang(n); N = 2 ** n
    sq = np.array_equal(A @ A, n * np.eye(N, dtype=int))
    adj = np.array_equal(np.abs(A), cube(n))
    odd = all((A[x, x ^ (1 << i)] * A[x ^ (1 << i), x ^ (1 << i) ^ (1 << j)] *
               A[x ^ (1 << i) ^ (1 << j), x ^ (1 << j)] * A[x ^ (1 << j), x]) == -1
              for x in range(N) for i in range(n) for j in range(i + 1, n))
    print(f"n={n}: A^2 = nI {sq}, |A| = adjacency of Q_n {adj}, every 4-cycle has product -1 {odd}")

def best_subgraph(n):
    N = 2 ** n; Q = cube(n); best, examples = None, []
    for S in itertools.combinations(range(N), N // 2 + 1):
        d = Q[np.ix_(S, S)].sum(0).max()
        if best is None or d < best: best, examples = d, []
        if d == best: examples.append(S)
    return best, examples

for n in range(1, 5):
    b, ex = best_subgraph(n)
    print(f"n={n}: least max degree on 2^(n-1)+1 vertices = {b}  (sqrt n = {math.sqrt(n):.3f}),"
          f" {len(ex)} optimal sets")
b, ex = best_subgraph(4)
S = ex[0]; Q4 = cube(4); deg = Q4[np.ix_(S, S)].sum(0)
print("an optimal set in Q4:", [format(v, "04b") for v in S], " degrees:", deg.tolist())

n=1: A^2 = nI True, |A| = adjacency of Q_n True, every 4-cycle has product -1 True
n=2: A^2 = nI True, |A| = adjacency of Q_n True, every 4-cycle has product -1 True
n=3: A^2 = nI True, |A| = adjacency of Q_n True, every 4-cycle has product -1 True
n=4: A^2 = nI True, |A| = adjacency of Q_n True, every 4-cycle has product -1 True
n=5: A^2 = nI True, |A| = adjacency of Q_n True, every 4-cycle has product -1 True
n=6: A^2 = nI True, |A| = adjacency of Q_n True, every 4-cycle has product -1 True
n=7: A^2 = nI True, |A| = adjacency of Q_n True, every 4-cycle has product -1 True
n=1: least max degree on 2^(n-1)+1 vertices = 1  (sqrt n = 1.000), 1 optimal sets
n=2: least max degree on 2^(n-1)+1 vertices = 2  (sqrt n = 1.414), 4 optimal sets
n=3: least max degree on 2^(n-1)+1 vertices = 2  (sqrt n = 1.732), 24 optimal sets


n=4: least max degree on 2^(n-1)+1 vertices = 2  (sqrt n = 2.000), 48 optimal sets


an optimal set in Q4: ['0000', '0001', '0010', '0101', '0110', '1011', '1100', '1101', '1110']  degrees: [2, 2, 2, 2, 2, 0, 2, 2, 2]


For $n=4$ the bound $\sqrt4=2$ is attained by $48$ sets, each an induced $8$-cycle plus an isolated vertex, as in the solution of Exercise 8.11. For $n=2$ and $n=3$ the least maximum degree is $\lceil\sqrt n\rceil=2$, so Huang's bound, after rounding up, is sharp there too.

### P8.7
**How to find it.** A copy of the Petersen graph is $\pi P\pi^{-1}$ for a permutation $\pi$. Draw random permutations until the copy is edge-disjoint from $P$; the eigenspace for $1$ is the null space of $A-I$.

In [9]:
P0 = petersen(); n = 10; K10 = np.ones((n, n), int) - np.eye(n, dtype=int)

def eigenspace(A, lam):
    w, V = np.linalg.eigh(A)
    return V[:, np.abs(w - lam) < 1e-8]

pet_spec = spectrum(P0); count = 0; third = 0; worst = 0.0; dims = Counter(); leftovers = Counter()
while count < 300:
    p = rng.permutation(n); P1 = P0[np.ix_(p, p)]
    if np.any(P0 & P1): continue
    count += 1
    U, W = eigenspace(P0, 1), eigenspace(P1, 1)
    dims[10 - int(np.linalg.matrix_rank(np.hstack([U, W]), tol=1e-8))] += 1
    rest = K10 - P0 - P1
    leftovers[tuple(sorted(rest.sum(0)))] += 1
    if spectrum(rest) == pet_spec: third += 1
    x = np.linalg.svd(np.hstack([U, -W]))[2][-1]; x = U @ x[:U.shape[1]]   # a common vector
    worst = max(worst, np.abs(rest @ x + 3 * x).max() / np.abs(x).max())
print("edge-disjoint pairs tested:", count)
print("dimension of the intersection of the two 1-eigenspaces:", dict(dims))
print("leftover graphs are 3-regular in every case:", list(leftovers) == [(3,) * 10])
print("leftovers with the Petersen spectrum:", third)
print("on the common vector x, the leftover satisfies A x = -3x up to", f"{worst:.1e}")

edge-disjoint pairs tested: 300
dimension of the intersection of the two 1-eigenspaces: {1: 300}
leftover graphs are 3-regular in every case: True
leftovers with the Petersen spectrum: 0
on the common vector x, the leftover satisfies A x = -3x up to 2.2e-14


Both eigenspaces have dimension $5$ inside the $9$-dimensional space $\mathbf 1^\perp$, so they must share at least a line, and the computation shows that they do. On a common vector $x$ the leftover $R$ satisfies $A_Rx=(J-I)x-x-x=-3x$, and $-3$ is not an eigenvalue of the Petersen graph: the contradiction of Proposition 8.10, seen live. The leftover is always $3$-regular, which is why only eigenvalues, and not degrees, can tell it apart from the Petersen graph.